In [1]:
# Imports
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

notebook_dir = Path.cwd()
notebooks_dir = notebook_dir.parent
repo_root = notebooks_dir.parent

sys.path.insert(0, str(notebooks_dir))
sys.path.insert(0, str(repo_root / "SnowPyt-MechParams" / "src"))

from caaml_utils import parse_caaml_directory
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.models import Pit

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(items, **_kwargs):
        return items

In [2]:
# Parse snowpit data

# Parse all CAAML files, then convert raw SnowPylot pits to SnowPyt-MechParams pits.
data_dir = repo_root /"snowpylot-applications"/ "snowpits" / "2015_2025"

print(data_dir)

raw_pits = parse_caaml_directory(data_dir)
mech_pits = [Pit.from_snow_pit(pit) for pit in raw_pits]

# Keep the original name available for any later notebook cells that use raw SnowPylot pits.
all_pits = raw_pits

print(f'Loaded {len(raw_pits):,} snow pits')
print(f'Converted {len(mech_pits):,} pits to SnowPyt-MechParams objects')

/Users/marykate/Desktop/Snow/snowpylot-applications/snowpits/2015_2025


/Users/marykate/Desktop/Snow/snowpylot-applications/sp-app/lib/python3.13/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")


Loaded 50,278 snow pits
Converted 50,278 pits to SnowPyt-MechParams objects


In [3]:
# Create slabs based on layer of concern

all_slabs = []

for raw_pit, mech_pit in zip(raw_pits, mech_pits):
    for slab in mech_pit.create_slabs(weak_layer_def="layer_of_concern"):
        all_slabs.append(
            {
                "pit_id": mech_pit.pit_id,
                "near_avalanche": raw_pit.core_info.location.pit_near_avalanche,
                "near_avalanche_raw": raw_pit.core_info.location.pit_near_avalanche,
                "slab": slab,
                "slab_layers": slab.layers,
                "snow_pit": raw_pit,
                "mech_pit": mech_pit,
                "ECT_results": raw_pit.stability_tests.ECT if raw_pit.stability_tests else [],
            }
        )

print(f"Created {len(all_slabs):,} usable slabs based on layer of concern")

Created 34,690 usable slabs based on layer of concern


In [4]:
# Create numerical variables from slab properties

def nominal_value(value):
    """Return a plain finite float from a scalar or uncertainties value."""
    if value is None:
        return np.nan
    value = getattr(value, "nominal_value", value)
    try:
        value = float(value)
    except (TypeError, ValueError):
        return np.nan
    return value if np.isfinite(value) else np.nan


def cumulative_hhi_cm(slab):
    """Sum layer hand hardness index multiplied by layer thickness in cm."""
    total = 0.0
    for layer in slab.layers:
        hhi = nominal_value(layer.hand_hardness_index)
        thickness = nominal_value(layer.thickness)
        if np.isnan(hhi) or np.isnan(thickness):
            return np.nan
        total += hhi * thickness
    return total


engine = ExecutionEngine()
weight_methods = {
    "density": "kim_jamieson_table2",
    "slab_weight": "sum_layer_weight",
}
d11_methods = {
    "density": "kim_jamieson_table2",
    "elastic_modulus": "schottner",
    "poissons_ratio": "kochle",
    "D11": "weissgraeber_rosendahl",
}

feature_records = []

for slab_info in tqdm(all_slabs, desc="Creating slab features", unit="slab"):
    slab = slab_info["slab"]

    weight_result = engine.execute_single(slab, "slab_weight", weight_methods)
    slab_weight = (
        nominal_value(weight_result.slab.slab_weight)
        if weight_result is not None and weight_result.success
        else np.nan
    )

    d11_result = engine.execute_single(slab, "D11", d11_methods)
    bending_stiffness = (
        nominal_value(d11_result.slab.D11)
        if d11_result is not None and d11_result.success
        else np.nan
    )

    feature_records.append(
        {
            "pit_id": slab_info["pit_id"],
            "slab_id": slab.slab_id,
            "near_avalanche": slab_info["near_avalanche"],
            "near_avalanche_raw": slab_info["near_avalanche_raw"],
            "n_layers": len(slab.layers),
            "slab_thickness_cm": nominal_value(slab.total_thickness),
            "slab_weight_N_m2": slab_weight,
            "bending_stiffness_D11_N_mm": bending_stiffness,
            "cumulative_hhi_cm": cumulative_hhi_cm(slab),
        }
    )

slab_features_df = pd.DataFrame(feature_records)

slab_features_df["has_slab_thickness"] = slab_features_df["slab_thickness_cm"].notna()
slab_features_df["has_slab_weight"] = slab_features_df["slab_weight_N_m2"].notna()
slab_features_df["has_D11"] = slab_features_df["bending_stiffness_D11_N_mm"].notna()
slab_features_df["has_cumulative_hhi"] = slab_features_df["cumulative_hhi_cm"].notna()

coverage_summary = slab_features_df[
    ["has_slab_thickness", "has_slab_weight", "has_D11", "has_cumulative_hhi"]
].sum()

print(f"Created slab_features_df with {len(slab_features_df):,} rows")
coverage_summary


Created slab_features_df with 34,690 rows


has_slab_thickness    34690
has_slab_weight       14796
has_D11                1590
has_cumulative_hhi    25027
dtype: int64

In [5]:

# Stability Tests
# Propagation
# Num taps



In [6]:
# Scale and normalize variables

In [7]:
# PCA

In [8]:
# Cluster Analysis

In [9]:
# Results